# ADS-B time as a lower bound

If an aircraft is seen moving on the ground before its take-off, it must have left the stand before that, so the
taxi time should be at least the ADS-B time `MVT − movement start`. That would be a hard lower bound like the LOBT
window. This notebook checks whether the 2025 labels respect it and what it would do to the out-of-fold predictions
of the final pipeline.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "pyproject.toml").is_file())
FEATURES = ROOT / "data/03_features"
sys.path.insert(0, str(ROOT / "model"))
import runner

runner.configure_tracking()
CORRECTOR_RUN = "295"   # run of 09_corrector

In [2]:
oof = pd.read_parquet(runner.artifact_dir(runner.resolve_run(CORRECTOR_RUN)) / "oof_predictions.parquet")
adsb = pd.read_parquet(FEATURES / "train_DEP_merged_sorted.parquet",
                       columns=["SCHED_TIME_UTC_mvt", "adsb_movement_to_mvt_sec", "adsb_movement_to_gate_direct_m"])
adsb = adsb.loc[pd.to_datetime(adsb["SCHED_TIME_UTC_mvt"], utc=True).dt.year.eq(2025)].reset_index(drop=True)
oof["adsb_sec"] = adsb["adsb_movement_to_mvt_sec"].to_numpy()[oof["row_id"]]
oof["adsb_to_stand_m"] = adsb["adsb_movement_to_gate_direct_m"].to_numpy()[oof["row_id"]]
seen = oof.loc[oof["adsb_sec"] > 0].copy()   # movement starts before the take-off
print(f"{len(oof):,} out-of-fold departures, {len(seen) / len(oof):.1%} with an ADS-B movement before the take-off")

2,085,041 out-of-fold departures, 53.4% with an ADS-B movement before the take-off


### Does the label respect the bound?

Share of these flights whose label is shorter than the ADS-B time by more than a tolerance.

In [3]:
pd.Series({f"label < ADS-B time − {tol} s": (seen["target"] < seen["adsb_sec"] - tol).mean() * 100
           for tol in [0, 30, 60, 120]}, name="% of flights").round(2).to_frame()

,% of flights
label < ADS-B time − 0 s,8.17
label < ADS-B time − 30 s,2.31
label < ADS-B time − 60 s,1.21
label < ADS-B time − 120 s,0.81


In [4]:
seen["below"] = seen["target"] < seen["adsb_sec"] - 120
by_airport = seen.groupby("ADEP_mvt")["below"].agg(flights="size", below="sum")
by_airport["% below"] = (by_airport["below"] / by_airport["flights"] * 100).round(2)
by_airport

,flights,below,% below
ADEP_mvt,,,
EDDF,201844,549,0.27
EDDM,135662,1072,0.79
EGLL,47857,96,0.20
EHAM,235496,3921,1.66
LEBL,170066,1467,0.86
LEMD,135000,628,0.47
LFPG,6743,48,0.71
LIRF,62003,135,0.22
LSZH,118565,803,0.68


### What these flights look like

For the flights more than two minutes below the bound, the detected movement starts long before the take-off, but the
recorded off-block comes much later.

In [5]:
below = seen.loc[seen["below"]]
pd.DataFrame({"ADS-B time (s)": below["adsb_sec"].describe(percentiles=[.1, .5, .9]),
              "label (s)": below["target"].describe(percentiles=[.1, .5, .9])}).round(0)

,ADS-B time (s),label (s)
count,8982.0,8982.0
mean,1831.0,621.0
std,2435.0,375.0
min,141.0,-12.0
10%,440.0,176.0
50%,980.0,594.0
90%,4249.0,1076.0
max,21596.0,3813.0


### As a hard bound on the final predictions

The out-of-fold predictions are raised to the ADS-B time where they are below it, in three variants: always, only for
ADS-B times up to 30 min, and additionally only for movements that start within 100 m of the stand. RMSE per month
(final = without the bound).

In [6]:
variants = {
    "always": seen.index,
    "≤ 30 min": seen.index[seen["adsb_sec"] <= 1800],
    "≤ 30 min, near stand": seen.index[(seen["adsb_sec"] <= 1800) & (seen["adsb_to_stand_m"] < 100)],
}
rmse = lambda values: ((values - oof["target"]) ** 2).groupby(oof["month"]).mean() ** 0.5
table = pd.DataFrame({"final": rmse(oof["final"])})
for name, rows in variants.items():
    bounded = oof["final"].copy()
    bounded[rows] = np.maximum(bounded[rows], oof.loc[rows, "adsb_sec"])
    table[name] = rmse(bounded)
table.loc["all"] = [np.sqrt((table[c] ** 2 * oof["month"].value_counts().sort_index()).sum() / len(oof)) for c in table]
table.round(2)

,final,always,≤ 30 min,"≤ 30 min, near stand"
month,,,,
1,327.95,370.68,328.21,328.16
2,218.85,365.12,219.19,219.02
3,181.53,227.03,181.81,181.72
4,182.50,231.23,182.75,182.75
5,274.95,290.51,275.00,275.03
6,294.33,338.97,294.70,294.62
7,270.95,333.44,271.09,271.05
8,223.69,324.31,223.87,223.80
9,229.80,272.88,230.16,230.02


### What follows for the models

The bound does not hold: for some of the flights with a movement before the take-off the label is clearly shorter,
mostly at the airports where ADS-B sees the ground well (EHAM above all; the few LTFM movements are almost all wrong).
The data do not show why the detected movement starts so long before the recorded off-block; towing, position noise
that looks like movement, a wrongly assigned trajectory or gaps in the positions are all possible. As a hard bound the
ADS-B time makes the predictions worse in every month, even when it is limited to short movements that start at the
stand.

The ADS-B time therefore stays a feature: the models learn that it is usually a lower bound and when it is not.

The movement start is the first time the aircraft moves faster than 2 kn for at least 30 s (`04_adsb_movement`). A
stricter definition, such as a higher speed threshold, might make the ADS-B time more reliable; there was no time left
to test this.